# Klasifikasi 18 kelas — YOLOv8 di Kaggle (CC0 + `Clothes_Dataset` digabung)

Notebook ini melatih **classifier Gayaku** dengan **Ultralytics YOLOv8** (`yolov8n-cls.pt`, 224 px)
di Kaggle, pada **dataset gabungan 18 kelas**: 15 folder `ryanbadai/clothes-dataset` ditambah tiga
kelas yang hanya dimiliki dataset CC0 (`Kaos_Lengan_Panjang`, `Sepatu`, `Topi`).

Penggabungannya **tidak** dikerjakan ulang di sini. Notebook memanggil
`ml_workspace/scripts/merge_datasets.py` dari repo, sehingga aturan dedupe, pemetaan label dan
pembagian split hanya punya satu sumber kebenaran. Alasannya ada di
`docs/adr/0008-merge-the-two-training-sources.md`: kedua dataset berbagi 1.401 foto yang sama
(hanya beda ukuran), jadi menggabungkannya tanpa dedupe akan menaruh foto yang sama di train dari
satu sumber dan di test dari sumber lain.

### Menyiapkan sesi Kaggle

1. Panel kanan → **Add Input** → `ryanbadai/clothes-dataset`.
2. Panel kanan → **Accelerator → GPU T4** (atau P100).
3. Panel kanan → **Internet → On**. Tanpa internet, `pip install ultralytics`, clone repo, dan
   clone dataset CC0 semuanya gagal, dan langkah 1–2 mengatakannya terus terang.
4. Jalankan sel dari atas ke bawah.

### Batas Kaggle yang perlu kamu tahu

* Kuota GPU gratis ±**30 jam/pekan**; satu sesi maksimal **12 jam**.
* `/kaggle/working` disimpan sebagai output (maks **20 GB**). Notebook ini menaruh pohon data
  (symlink) dan checkpoint di `/kaggle/temp`, dan hanya artefak + zip di `/kaggle/working`.
* **Save & Run All (Commit)** berjalan di server (tab boleh ditutup) tetapi memakai kuota GPU:
  iterasi interaktif dulu, commit sekali di akhir.

### Kontrak aplikasi (tidak boleh berubah)

| | |
| --- | --- |
| input | float32 (`float16` ditoleransi), NHWC `[1,224,224,3]` atau NCHW `[1,3,224,224]`, RGB `[0,1]`; normalisasi harus ter-*bake* di dalam model |
| output | satu tensor `[1,18]` float32, distribusi probabilitas (jumlah ≈ 1) |
| urutan kelas | urutan `labels` di `classes.json` = `model.names` = urutan abjad nama folder |

Yang menegakkan: `flutter_app/lib/logic/scan_pipeline.dart::checkContracts` saat model dimuat, dan
`ml_workspace/scripts/gen_dart_labels.py --model` saat codegen. Aset dan daftar label Dart harus
diganti dalam **satu commit**.

### Bacaan terkait

* `docs/adr/0008-merge-the-two-training-sources.md` — kenapa digabung, aturan dedupe, `--prefer-source`.
* `docs/adr/0007-carousell-clothes-dataset.md` — dataset Carousell: 15 kelas, tanpa split, noise label.
* `docs/adr/0006-yolov8-classifier-agpl.md` — pilihan YOLOv8 dan konsekuensi **AGPL-3.0**.
* `docs/adr/0005-classify-the-cutout.md` — satu pakaian per foto adalah syarat, bukan preferensi.
* `ml_workspace/yolo/README.md` — langkah integrasi setelah zip diunduh.

## 1. Runtime, GPU, dan instalasi

Sel ini memastikan GPU terlihat, memasang `ultralytics`, dan mencetak versi Python/PyTorch/
ultralytics. Versi dicetak karena ultralytics mengubah API antar rilis: angka versi adalah cara
membedakan "bug kita" dari "perubahan mereka" ketika hasil berbeda dari `report.md` orang lain.

Kalau CUDA tidak terlihat, accelerator belum dipasang — panel kanan → **Accelerator: GPU T4**.
Notebook tetap jalan di CPU, tetapi 30 epoch pada ±9.700 foto di CPU memakan berjam-jam.

In [ ]:
# Langkah 1/8 — cek runtime dan pasang ultralytics.
# Kalau sel ini error "Could not find a version that satisfies the requirement" / koneksi macet:
# panel kanan -> Internet -> On, lalu jalankan ulang sel ini.
!pip install -q ultralytics

import importlib.util
import sys

import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")

GPU = torch.cuda.is_available()
if GPU:
    DEVICE = 0
    print(f"cuda          {torch.version.cuda}")
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("cuda          TIDAK ADA")
    print("              panel kanan -> Accelerator -> GPU T4, lalu jalankan ulang dari sel ini")

# Langkah 6 memverifikasi .tflite dengan runtime yang terpisah dari exporter-nya.
# Kaggle sudah menyediakan TensorFlow; ai_edge_litert ikut terpasang bersama litert-torch saat export.
for module in ("tensorflow", "ai_edge_litert"):
    print(f"{module:<14}{'tersedia' if importlib.util.find_spec(module) else 'belum terpasang'}")

## 2. Bangun dataset gabungan lewat `merge_datasets.py`

Tiga hal yang dilakukan sel berikutnya:

1. **Clone repo ini** (default `main`; ganti `REPO_REF` ke nama branch PR kalau skripnya belum
   di-merge) — `merge_datasets.py` ada di sana, dan tabel labelnya `ml_workspace/labels.json`.
2. **Clone dataset CC0** `alexeygrigorev/clothing-dataset-small` (±103 MB). Dataset ini tidak ada di
   Kaggle sebagai input, dan justru dialah yang membawa `Sepatu` dan `Topi`.
3. **Panggil skrip penggabungan.** Skrip melakukan dedupe (sha256 + dHash 64-bit + signature
   16×16), memetakan 10 folder CC0 ke 18 kelas Indonesia, membagi train/val/test per folder
   sumber, lalu menulis `manifest.csv`, `conflicts.csv`, `merged_dataset.json`, dan
   `classes_merged.json`.

Foto yang sama di dua sumber hanya disimpan sekali: salinan `Clothes_Dataset` yang menang (resolusi
720×1280, label yang sama dengan kosakata UI), sesuai `--prefer-source clothes` di ADR-0008. Foto
identik yang di dalam `Clothes_Dataset` sendiri terdaftar di dua kelas berbeda dibuang seluruhnya
dan dicatat di `conflicts.csv`.

Kalau sel ini berhenti dengan `no Kaggle token` atau error lain dari skrip, jangan lanjutkan:
tanpa pohon data yang lolos assert, angka di bawah tidak berarti apa-apa.

In [ ]:
# Langkah 2/8 — clone repo + dataset CC0, lalu jalankan merge_datasets.py.
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "main"          # ganti ke nama branch PR kalau skripnya belum di-merge
CC0_URL = "https://github.com/alexeygrigorev/clothing-dataset-small.git"

WORK_DIR = Path("/kaggle/temp") if os.access("/kaggle/temp", os.W_OK) else Path("/kaggle/working/temp")
WORK_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR = Path("/kaggle/working/artifacts")   # inilah yang ikut tersimpan
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR = WORK_DIR / "merged"
REPO_DIR = WORK_DIR / "gayaku"
CC0_DIR = WORK_DIR / "clothing-dataset-small"


def run(command, **kwargs):
    print("+", " ".join(str(part) for part in command))
    completed = subprocess.run(command, text=True, **kwargs)
    if completed.returncode != 0:
        raise SystemExit(f"perintah gagal (exit {completed.returncode}): {' '.join(map(str, command))}")
    return completed


def clone(url, target):
    if (target / ".git").is_dir():
        print(f"memakai ulang {target}")
        return
    shutil.rmtree(target, ignore_errors=True)
    run(["git", "clone", "--depth", "1", url, str(target)])


DOCUMENTED_ROOT = Path("/kaggle/input/clothes-dataset/Clothes_Dataset")


def find_clothes_root():
    if DOCUMENTED_ROOT.is_dir():
        return DOCUMENTED_ROOT
    for candidate in sorted(Path("/kaggle/input").rglob("Clothes_Dataset")):
        if candidate.is_dir():
            print(f"catatan: {DOCUMENTED_ROOT} tidak ada, memakai {candidate}")
            return candidate
    return None


CLOTHES_DIR = find_clothes_root()
if CLOTHES_DIR is None:
    raise SystemExit(
        "Dataset ryanbadai/clothes-dataset tidak ditemukan.\n"
        "  Panel kanan -> Add Input -> 'ryanbadai/clothes-dataset', lalu jalankan ulang sel ini.\n"
        f"  Path yang dicari: {DOCUMENTED_ROOT}"
    )

clone(REPO_URL, REPO_DIR)
if REPO_REF != "main":
    run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF])
    run(["git", "-C", str(REPO_DIR), "checkout", "FETCH_HEAD"])

MERGE_SCRIPT = REPO_DIR / "ml_workspace/scripts/merge_datasets.py"
if not MERGE_SCRIPT.is_file():
    raise SystemExit(
        f"{MERGE_SCRIPT} tidak ada di ref '{REPO_REF}'.\n"
        "  Kalau branch PR belum di-push, push dulu; kalau sudah, set REPO_REF ke nama branch itu."
    )

clone(CC0_URL, CC0_DIR)

CLASSES_JSON_FROM_MERGE = ARTIFACTS_DIR / "classes_merged.json"
run(
    [
        sys.executable, str(MERGE_SCRIPT),
        "--cc0-dir", str(CC0_DIR),
        "--clothes-dir", str(CLOTHES_DIR),
        "--out", str(DATASET_DIR),
        "--summary", str(ARTIFACTS_DIR / "merged_dataset.json"),
        "--classes-out", str(CLASSES_JSON_FROM_MERGE),
        "--force",
    ],
)

## 3. Periksa pohon data sebelum menghabiskan kuota GPU

Sel ini tidak melatih apa pun. Ia memeriksa lima hal, dan berhenti kalau salah satunya gagal:

* `classes_merged.json` berisi **18** label;
* setiap split (`train`, `val`, `test`) punya **semua** kelas — kelas yang tidak ada di test tidak
  bisa dilaporkan recall-nya;
* jumlah foto di pohon sama dengan `totals.kept` di `merged_dataset.json` — kalau berbeda, ada link
  yang gagal dibuat;
* semua assert di `merged_dataset.json` berawalan `passed`;
* `manifest.csv` memang ada, supaya keputusan dedupe bisa diaudit setelah run ini selesai.

Tabel yang dicetak adalah **jumlah foto unik per kelas setelah dedupe**, bukan angka 500 yang
tertulis di halaman dataset.

In [ ]:
# Langkah 3/8 — periksa pohon gabungan dan cetak jumlah unik per kelas.
import csv
import json
from pathlib import Path

SUMMARY = json.loads((ARTIFACTS_DIR / "merged_dataset.json").read_text(encoding="utf-8"))
CLASSES = json.loads(CLASSES_JSON_FROM_MERGE.read_text(encoding="utf-8"))["labels"]
SPLITS = ("train", "val", "test")

if len(CLASSES) != 18:
    raise SystemExit(f"classes_merged.json berisi {len(CLASSES)} label, seharusnya 18: {CLASSES}")
if CLASSES != sorted(CLASSES):
    raise SystemExit("classes_merged.json tidak terurut abjad; urutan itu yang dipakai ultralytics")

counts = {}
for split in SPLITS:
    folders = sorted(folder.name for folder in (DATASET_DIR / split).iterdir() if folder.is_dir())
    if folders != CLASSES:
        raise SystemExit(f"folder kelas di split {split} tidak sama dengan daftar label: {folders}")
    counts[split] = {
        label: len([p for p in (DATASET_DIR / split / label).iterdir() if p.is_file() or p.is_symlink()])
        for label in CLASSES
    }
    if any(value == 0 for value in counts[split].values()):
        kosong = [label for label, value in counts[split].items() if value == 0]
        raise SystemExit(f"split {split} tidak punya foto untuk kelas: {kosong}")

tree_total = sum(sum(values.values()) for values in counts.values())
if tree_total != SUMMARY["totals"]["kept"]:
    raise SystemExit(
        f"pohon berisi {tree_total} foto tetapi merge melaporkan {SUMMARY['totals']['kept']} unik"
    )

gagal = {key: value for key, value in SUMMARY["assertions"].items() if not value.startswith("passed")}
if gagal:
    raise SystemExit(f"assert merge tidak lolos: {gagal}")

MANIFEST = DATASET_DIR / "manifest.csv"
for needed in (MANIFEST, DATASET_DIR / "conflicts.csv"):
    if not needed.is_file():
        raise SystemExit(f"{needed} tidak ada — jalankan ulang langkah 2")

print(f"dataset       {DATASET_DIR}")
print(f"sumber        {', '.join(sorted(SUMMARY['sources']))}")
print(f"{'kelas':<22}{'train':>7}{'val':>6}{'test':>6}{'unik':>7}  sumber")
for label in CLASSES:
    row = SUMMARY["per_class"][label]
    print(
        f"{label:<22}{counts['train'][label]:>7}{counts['val'][label]:>6}{counts['test'][label]:>6}"
        f"{counts['train'][label] + counts['val'][label] + counts['test'][label]:>7}  "
        f"{', '.join(f'{k} {v}' for k, v in row['by_source'].items())}"
    )
print(f"{'TOTAL':<22}{sum(counts['train'].values()):>7}{sum(counts['val'].values()):>6}"
      f"{sum(counts['test'].values()):>6}{tree_total:>7}")
print()
print("alasan dedupe (dari manifest):")
for reason, jumlah in sorted(SUMMARY["totals"]["by_reason"].items(), key=lambda item: -item[1]):
    print(f"  {reason:<40}{jumlah:>6}")
print()
for key, value in SUMMARY["assertions"].items():
    print(f"  {key:<32}{value}")

## 4. Latihan

`yolov8n-cls.pt` adalah checkpoint klasifikasi nano (beberapa MB, sudah terlatih di ImageNet).
±9.700 foto dengan 30 epoch selesai dalam puluhan menit di GPU Kaggle. Ini klasifikasi, bukan
deteksi: tidak ada bounding box, tidak ada NMS, grafnya hanya satu input gambar dan satu output
vektor probabilitas.

* `EPOCHS` (30), `BATCH` (64), `SEED_TRAIN` (42), `IMGSZ` (224 — kontrak aplikasi);
* `PATIENCE = 10` berhenti lebih awal alih-alih membakar kuota;
* `workers=2` karena mesin Kaggle memberi 2–4 vCPU;
* hasil masuk `/kaggle/temp/runs`, bukan `/kaggle/working`, supaya checkpoint tidak jadi output.

`best.pt` dipakai semua langkah berikutnya, dan urutan kelasnya harus sama dengan urutan folder —
kalau tidak, `classes.json` tidak berarti apa-apa.

In [ ]:
# Langkah 4/8 — latih yolov8n-cls pada 224 px.
from pathlib import Path

from ultralytics import YOLO

MODEL = "yolov8n-cls.pt"   # coba "yolo11n-cls.pt" atau "yolov8s-cls.pt" untuk backbone yang lebih besar
EPOCHS = 30
BATCH = 64
SEED_TRAIN = 42            # seed latihan (SEED di langkah 3 mengatur pembagian data)
IMGSZ = 224                # kontrak aplikasi
PATIENCE = 10
RUN_NAME = "yolov8n-cls-merged-224"

model = YOLO(MODEL)
final_metrics = model.train(
    data=str(DATASET_DIR),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED_TRAIN,
    patience=PATIENCE,
    device=DEVICE,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,   # jalankan ulang berarti menimpa run ini, bukan membuat run2, run3 ...
    pretrained=True,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
CLASS_NAMES = [model.names[index] for index in range(len(model.names))]

print(f"run           {RUN_DIR}")
print(f"bobot terbaik {BEST_WEIGHTS} ({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print(f"metrik akhir  {final_metrics}")
print(f"{len(CLASS_NAMES)} kelas dalam urutan model: {CLASS_NAMES}")
if CLASS_NAMES != CLASSES:
    raise SystemExit("urutan kelas model berbeda dari folder dataset — model ini jangan dipakai")

## 5. Evaluasi: `val` untuk memilih epoch, `test` untuk dilaporkan

Split **`val`** dipakai ultralytics untuk memilih `best.pt`, jadi angkanya sedikit optimistis.
Split **`test`** tidak pernah dilihat model dan tidak dipakai memilih epoch — itulah angka yang
dilaporkan, bersama confusion matrix dan recall per kelas.

Sel ini melakukan dua hal untuk setiap split:

1. `model.val(...)`, yang menghasilkan confusion matrix ultralytics dan grafiknya;
2. satu pass `model.predict` atas folder split untuk mendapatkan label per foto, dipakai
   `sklearn.classification_report` (precision/recall/F1 per kelas) dan `per_class_accuracy.csv`.

Confusion matrix dari dua jalur itu dibandingkan: selisih sel yang tidak nol berarti pipeline
evaluasi kita berbeda dari ultralytics, bukan modelnya salah. Idealnya 0.

In [ ]:
# Langkah 5/8 — evaluasi pada val DAN test.
import csv as csv_module
import shutil as shutil_module

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix as sklearn_confusion_matrix
from ultralytics import YOLO

trained = YOLO(str(BEST_WEIGHTS))

TEST_SPLIT = "test"   # dasar gerbang fidélitas di langkah 6; tidak dipakai memilih epoch
MEASURED = {}
for split in ("val", "test"):
    metrics = trained.val(
        data=str(DATASET_DIR),
        split=split,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        plots=True,
        project=str(RUNS_DIR),
        name=f"val-{split}",
        exist_ok=True,
    )
    ULTRA_MATRIX = np.asarray(metrics.confusion_matrix.matrix)   # [predicted][true]
    results = trained.predict(
        source=str(DATASET_DIR / split), imgsz=IMGSZ, batch=BATCH, device=DEVICE, verbose=False
    )
    y_true = [Path(result.path).parent.name for result in results]
    y_pred = [CLASSES[int(result.probs.top1)] for result in results]
    REPORT_TEXT = classification_report(
        y_true, y_pred, labels=CLASSES, target_names=CLASSES, digits=4, zero_division=0
    )
    REPORT_DICT = classification_report(
        y_true, y_pred, labels=CLASSES, target_names=CLASSES, digits=4, zero_division=0,
        output_dict=True,
    )
    SKLEARN_MATRIX = sklearn_confusion_matrix(y_true, y_pred, labels=CLASSES)   # [true][predicted]
    beda = int((ULTRA_MATRIX != SKLEARN_MATRIX.T).sum())

    MEASURED[split] = {
        "top1": float(metrics.top1),
        "top5": float(metrics.top5),
        "accuracy": float(REPORT_DICT["accuracy"]),
        "macro_f1": float(REPORT_DICT["macro avg"]["f1-score"]),
        "per_class": REPORT_DICT,
        "matrix": SKLEARN_MATRIX,
        "disagreement": beda,
        "images": len(results),
    }

    with (ARTIFACTS_DIR / f"classification_report_{split}.txt").open("w", encoding="utf-8") as handle:
        handle.write(REPORT_TEXT)
    with (ARTIFACTS_DIR / f"classification_report_{split}.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv_module.writer(handle)
        writer.writerow(("label", "precision", "recall", "f1-score", "support"))
        for row_label in CLASSES:
            row = REPORT_DICT[row_label]
            writer.writerow(
                (row_label, f"{row['precision']:.4f}", f"{row['recall']:.4f}",
                 f"{row['f1-score']:.4f}", int(row["support"]))
            )

    SUPPORT = SKLEARN_MATRIX.sum(axis=0)
    CORRECT = np.diag(SKLEARN_MATRIX)
    with (ARTIFACTS_DIR / f"per_class_accuracy_{split}.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv_module.writer(handle)
        writer.writerow(("label", "support", "correct", "recall"))
        for label in CLASSES:
            index = CLASSES.index(label)
            writer.writerow((label, int(SUPPORT[index]), int(CORRECT[index]),
                             f"{CORRECT[index] / max(1.0, SUPPORT[index]):.4f}"))

    with (ARTIFACTS_DIR / f"confusion_matrix_{split}.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv_module.writer(handle)
        writer.writerow(("predicted \\ true", *CLASSES))
        for index, label in enumerate(CLASSES):
            writer.writerow((label, *[int(value) for value in ULTRA_MATRIX[index]]))

    for png in sorted(Path(metrics.save_dir).glob("confusion_matrix*.png")):
        target = ARTIFACTS_DIR / f"{png.stem}_{split}{png.suffix}"
        shutil_module.copy2(png, target)
        print(f"grafik        {target}")

    print(
        f"{split:<5} top-1 {MEASURED[split]['top1']:.4f}  top-5 {MEASURED[split]['top5']:.4f}  "
        f"akurasi {MEASURED[split]['accuracy']:.4f}  macro-F1 {MEASURED[split]['macro_f1']:.4f}  "
        f"{MEASURED[split]['images']} foto  silang-cek {beda} sel berbeda"
    )

TEST_TOP1 = MEASURED["test"]["top1"]
TEST_MACRO_F1 = MEASURED["test"]["macro_f1"]

### Recall per kelas pada split `test`

Angka di `per_class_accuracy_test.csv` adalah bagian yang wajib dibaca: akurasi total bisa tinggi
sementara satu kelas hampir selalu salah. Dua tempat yang paling mungkin bermasalah, dan sudah
diperkirakan sebelum latihan:

* **`Jaket` vs `Jaket_Denim` vs `Jaket_Olahraga` vs `Mantel` vs `Blazer` vs `Sweter`** — di dataset
  CC0 semua ini satu kelas, `outwear`; di `Clothes_Dataset` mereka dipisah. Foto CC0 yang masuk
  `Jaket` karena itu bersaing dengan lima kelas yang lebih spesifik, jadi `Jaket` cenderung
  menyerap semuanya.
* **`Kaos_Lengan_Panjang` vs `Sweter`** — 403 foto CC0 `longsleeve` diberi label `Sweter` oleh
  `Clothes_Dataset`, jadi kelas ini hanya tersisa 172 foto. Kalau recall-nya jatuh, penyebabnya
  bisa data, bukan model; itu sebabnya `merge_datasets.py --prefer-source cc0` ada.

## 6. Ekspor ke `.tflite`, lalu verifikasi

**Bagian 1** mengekspor graf: `format="litert"` adalah nama sekarang untuk ekspor LiteRT
(menghasilkan `.tflite`). Untuk klasifikasi tidak ada NMS maupun decoding box di dalam graf, dan
ekspor dijalankan di CPU karena tracing tidak butuh GPU.

**Bagian 2** yang menentukan file ini boleh dipakai, dan sengaja paranoid — model yang salah tetapi
*masih bisa dimuat* lebih berbahaya daripada yang gagal dimuat:

* verifikasi memakai **runtime independen** (`tf.lite.Interpreter` atau `ai_edge_litert`), bukan
  exporter yang menulis filenya; yang diperiksa sama dengan `scan_pipeline.dart::checkContracts`:
  input float32 batch 1 `[1,224,224,3]` atau `[1,3,224,224]`, output `[1,18]` dengan jumlah ≈ 1;
* kalau output ternyata logits (jumlah ≠ 1), sel mengekspor ulang dengan Softmax di ujungnya;
* **gerbang fidélitas**: ≥100 foto dari split `test` dilewatkan ke PyTorch dan ke `.tflite` dengan
  piksel yang sama, lalu dicetak berapa argmax yang setuju. Gerbangnya sama dengan
  `ml_workspace/bootstrap/export_tflite.py tflite`: akurasi `.tflite` ≥ 0,50 dan tidak lebih dari
  0,10 di bawah PyTorch.

In [ ]:
# Langkah 6/8 (bagian 1) — ekspor ke LiteRT (.tflite).
import hashlib

from ultralytics import YOLO

TFLITE_PATH = ARTIFACTS_DIR / "garment_classifier.tflite"
EXPORT_STEPS = []

trained = YOLO(str(BEST_WEIGHTS))
exported_path = Path(trained.export(format="litert", imgsz=IMGSZ, device="cpu"))
shutil.copy2(exported_path, TFLITE_PATH)
EXPORT_STEPS.append(f"YOLO('{BEST_WEIGHTS}').export(format='litert', imgsz={IMGSZ}, device='cpu')")

print(f"diekspor     {exported_path}")
print(f"disalin ke   {TFLITE_PATH} ({TFLITE_PATH.stat().st_size / 1e6:.2f} MB)")
print(f"sha256       {hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()}")

In [ ]:
# Langkah 6/8 (bagian 2) — verifikasi kontrak aplikasi, lalu bandingkan dengan PyTorch.
import importlib.util
import math
import shutil
from collections import namedtuple

import numpy as np
import torch
from PIL import Image

Contract = namedtuple(
    "Contract", "runtime interpreter in_index in_shape in_dtype out_index out_shape out_dtype layout"
)


def load_interpreter(path):
    """Runtime TFLite yang independen dari exporter yang menulis filenya."""
    if importlib.util.find_spec("tensorflow"):
        import tensorflow as tf

        return tf.lite.Interpreter(model_path=str(path)), f"tensorflow {tf.__version__} tf.lite.Interpreter"
    from ai_edge_litert.interpreter import Interpreter

    return Interpreter(model_path=str(path)), "ai_edge_litert.interpreter.Interpreter"


def inspect_model(path):
    """Baca IO tensor yang akan dilihat aplikasi, sekaligus layout-nya (NHWC atau NCHW)."""
    interpreter, runtime = load_interpreter(path)
    interpreter.allocate_tensors()
    inputs = interpreter.get_input_details()[0]
    outputs = interpreter.get_output_details()[0]
    shape = tuple(int(value) for value in inputs["shape"])
    if shape[1:] == (IMGSZ, IMGSZ, 3):
        layout = "NHWC"
    elif shape[1:] == (3, IMGSZ, IMGSZ):
        layout = "NCHW"
    else:
        raise SystemExit(f"bentuk input {shape} tidak dikenal: aplikasi menerima [1,224,224,3] atau [1,3,224,224]")
    return Contract(
        runtime,
        interpreter,
        inputs["index"],
        shape,
        inputs["dtype"],
        outputs["index"],
        tuple(int(value) for value in outputs["shape"]),
        outputs["dtype"],
        layout,
    )


def assert_app_contract(contract, class_count):
    """Persis yang ditolak flutter_app/lib/logic/scan_pipeline.dart."""
    assert contract.in_dtype == np.float32, f"dtype input {contract.in_dtype.__name__}: aplikasi mengirim float32"
    assert contract.out_dtype == np.float32, f"dtype output {contract.out_dtype.__name__}: aplikasi membaca float32"
    assert contract.in_shape[0] == 1 and contract.out_shape[0] == 1, "kedua tensor harus batch 1"
    assert contract.out_shape[1] == class_count, (
        f"model memberi {contract.out_shape[1]} skor tapi classes.json akan berisi {class_count}: "
        "aplikasi melempar pesan Indonesia yang menyebut ketidakcocokan ini"
    )


def predict(contract, pixels_hwc):
    """[H,W,3] float32 RGB [0,1] -> output model [N] float32."""
    array = pixels_hwc[None] if contract.layout == "NHWC" else np.transpose(pixels_hwc, (2, 0, 1))[None]
    contract.interpreter.set_tensor(contract.in_index, array.astype(np.float32))
    contract.interpreter.run()
    return contract.interpreter.get_tensor(contract.out_index)[0]


def load_pixels(path, size=IMGSZ):
    """Piksel identik untuk kedua model: RGB, sisi pendek di-resize, center crop, [0,1] float32.

    Ini pipeline evaluasi ultralytics sendiri (Resize -> CenterCrop -> ToTensor, mean 0 / std 1).
    """
    image = Image.open(path).convert("RGB")
    scale = size / min(image.size)
    image = image.resize(
        (max(size, round(image.width * scale)), max(size, round(image.height * scale))), Image.BILINEAR
    )
    left, top = (image.width - size) // 2, (image.height - size) // 2
    return np.asarray(image.crop((left, top, left + size, top + size)), dtype=np.float32) / 255.0


network = trained.model.eval()  # mode eval: kepala klasifikasi mengembalikan probabilitas, bukan logits


def torch_probabilities(pixels_hwc):
    """Jaringan PyTorch terlatih pada piksel yang sama — acuan yang harus disetujui hasil ekspor."""
    with torch.no_grad():
        output = network(torch.from_numpy(np.transpose(pixels_hwc, (2, 0, 1)))[None])
        probabilities = output[0] if isinstance(output, (tuple, list)) else output
        return probabilities[0].detach().cpu().numpy()


PROBE = load_pixels(sorted((DATASET_DIR / TEST_SPLIT / CLASSES[0]).iterdir())[0])

contract = inspect_model(TFLITE_PATH)
print(f"runtime       {contract.runtime}")
print(f"input         {contract.in_shape} {contract.in_dtype.__name__}")
print(f"output        {contract.out_shape} {contract.out_dtype.__name__}")
print(f"layout        {contract.layout} (aplikasi membaca bentuk tensor dan menyesuaikan diri)")
assert_app_contract(contract, len(CLASSES))

PROBE_SUM = float(predict(contract, PROBE).sum())
print(f"jumlah skor   {PROBE_SUM:.6f}")

if not 0.99 <= PROBE_SUM <= 1.01:
    print("graf mengembalikan logits, bukan probabilitas — mengekspor ulang dengan Softmax di ujungnya")

    for module in network.modules():
        if type(module).__name__ == "Classify":
            module.export = False  # di mode eval kepala lalu mengembalikan (probabilitas, logits); kita mau logits-nya


    class WithSoftmax(torch.nn.Module):
        """[1,3,224,224] RGB [0,1] -> [1,N] probabilitas softmax, yaitu kontrak aplikasi."""

        def __init__(self, network):
            super().__init__()
            self.network = network

        def forward(self, x):
            output = self.network(x)
            logits = output[1] if isinstance(output, (tuple, list)) else output
            return torch.nn.functional.softmax(logits, dim=1)


    from ultralytics.utils.export.litert import torch2litert

    target = TFLITE_PATH.with_name("garment_classifier_softmax.tflite")
    torch2litert(WithSoftmax(network), torch.zeros(1, 3, IMGSZ, IMGSZ), target, metadata={"task": "classify"})
    shutil.move(str(target), str(TFLITE_PATH))
    EXPORT_STEPS.append("torch2litert(WithSoftmax(best.pt), torch.zeros(1,3,224,224), ...) -> garment_classifier.tflite")

    contract = inspect_model(TFLITE_PATH)
    assert_app_contract(contract, len(CLASSES))
    PROBE_SUM = float(predict(contract, PROBE).sum())
    print(f"jumlah skor   {PROBE_SUM:.6f} (setelah ekspor ulang)")
    if not 0.99 <= PROBE_SUM <= 1.01:
        raise SystemExit("hasil ekspor tetap bukan distribusi probabilitas — jangan dipakai")

# Gerbang fidélitas: >= 100 foto held-out, piksel sama ke kedua model, bandingkan argmax-nya.
PER_CLASS = max(2, math.ceil(120 / len(CLASSES)))
held_out = []
for label in CLASSES:
    photos = sorted((DATASET_DIR / TEST_SPLIT / label).iterdir())
    step = max(1, len(photos) // PER_CLASS)
    held_out.extend((photo, label) for photo in photos[::step][:PER_CLASS])
if len(held_out) < 100:
    raise SystemExit(f"hanya {len(held_out)} foto held-out — pemeriksaan fidélitas butuh minimal 100")

rows = []
for photo, label in held_out:
    pixels = load_pixels(photo)
    expected = torch_probabilities(pixels)
    actual = predict(contract, pixels)
    rows.append(
        (
            label,
            CLASSES[int(expected.argmax())],
            CLASSES[int(actual.argmax())],
            float(actual.sum()),
            float(np.abs(expected - actual).max()),
        )
    )

IMAGE_COUNT = len(rows)
AGREE_COUNT = sum(1 for _, reference, converted, _, _ in rows if reference == converted)
TORCH_CORRECT = sum(1 for label, reference, _, _, _ in rows if label == reference)
TFLITE_CORRECT = sum(1 for label, _, converted, _, _ in rows if label == converted)
AGREEMENT = AGREE_COUNT / IMAGE_COUNT
TORCH_ACCURACY = TORCH_CORRECT / IMAGE_COUNT
TFLITE_ACCURACY = TFLITE_CORRECT / IMAGE_COUNT
MAX_SUM_ERROR = max(abs(row[3] - 1.0) for row in rows)
MAX_PROB_DIFF = max(row[4] for row in rows)
GATE_OK = TFLITE_ACCURACY >= 0.50 and TFLITE_ACCURACY >= TORCH_ACCURACY - 0.10

print(f"foto          {IMAGE_COUNT} held-out dari split {TEST_SPLIT}")
print(f"akurasi       pytorch {TORCH_ACCURACY:.4f}   .tflite {TFLITE_ACCURACY:.4f}")
print(f"kesesuaian    {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f} argmax identik")
print(f"maks |jml-1|  {MAX_SUM_ERROR:.3e}")
print(f"maks |beda|   {MAX_PROB_DIFF:.4f}")
print(f"gerbang       .tflite >= 0.50 dan >= pytorch - 0.10 -> {'LULUS' if GATE_OK else 'GAGAL'}")
if not GATE_OK:
    raise SystemExit(
        f"model hasil konversi mendapat {TFLITE_ACCURACY:.4f} sementara PyTorch {TORCH_ACCURACY:.4f} "
        "pada foto yang sama: konversinya rusak, jangan dipakai"
    )

## 7. Tulis `classes.json` dan `report.md`

`classes.json` memakai format repo — `{"version": 1, "source": ..., "labels": [...]}` — dan urutan
`labels` **adalah** urutan output model. Sel ini menulisnya dari `model.names` dan memastikan daftar
itu sama dengan `classes_merged.json` yang dibuat `merge_datasets.py`: kalau berbeda, ada folder
kelas yang berubah di antara langkah 2 dan 4, dan setiap prediksi di aplikasi akan bergeser satu
kelas.

`report.md` adalah jejak bukti: versi runtime, angka `val` dan `test`, precision/recall/F1 per kelas,
confusion matrix, hasil gerbang konversi, IO tensor, sha256 dan ukuran `.tflite`, perintah, catatan
noise label, catatan lisensi, dan pembanding yang jujur. **Yang dilaporkan adalah split `test`.**
`val` dipakai memilih epoch, jadi menyebut angka `val` sebagai hasil akhir itu menyesatkan.

In [ ]:
# Langkah 7/8 — tulis classes.json dan report.md.
import hashlib
import json
from datetime import datetime, timezone

CLASSES_JSON = ARTIFACTS_DIR / "classes.json"
if CLASSES != json.loads(CLASSES_JSON_FROM_MERGE.read_text(encoding="utf-8"))["labels"]:
    raise SystemExit("urutan kelas model berbeda dari classes_merged.json — model ini jangan dipakai")

CLASSES_JSON.write_text(
    json.dumps(
        {
            "version": 1,
            "source": (
                f"Ultralytics {MODEL} classification fine-tuned on merge_datasets.py output "
                f"(CC0 + ryanbadai/clothes-dataset, {len(CLASSES)} classes, imgsz {IMGSZ}, "
                f"{EPOCHS} epochs, batch {BATCH}, seed {SEED_TRAIN}); "
                "index = the model's output index (ultralytics' alphabetical class-folder order)"
            ),
            "labels": CLASSES,
        },
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)
print(f"ditulis       {CLASSES_JSON} ({len(CLASSES)} label)")

TFLITE_BYTES = TFLITE_PATH.stat().st_size
TFLITE_SHA256 = hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()
DATE_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
GATE_LABEL = "PASSED" if GATE_OK else "FAILED"
GPU_NAME = torch.cuda.get_device_name(0) if GPU else "none (CPU)"
TRAIN_IMAGES = sum(counts["train"].values())
VAL_IMAGES = sum(counts["val"].values())
TEST_IMAGES = sum(counts["test"].values())
CLASS_LIST = ", ".join(CLASSES)


def per_class_table(split):
    rows = MEASURED[split]["per_class"]
    lines = []
    for label in CLASSES:
        row = rows[label]
        lines.append(
            f"| {label} | {row['precision']:.4f} | {row['recall']:.4f} | "
            f"{row['f1-score']:.4f} | {int(row['support'])} |"
        )
    return "\n".join(lines)


def matrix_table(split):
    matrix = MEASURED[split]["matrix"]
    header = "| true \\ predicted | " + " | ".join(CLASSES) + " |"
    lines = [header, "| --- |" + " --- |" * len(CLASSES)]
    for index, label in enumerate(CLASSES):
        lines.append(f"| **{label}** | " + " | ".join(str(int(value)) for value in matrix[index]) + " |")
    return "\n".join(lines)


MAPPING_TABLE = "\n".join(
    f"| `{cc0}` | `{merged}` |" for cc0, merged in sorted(SUMMARY["mapping"]["cc0_to_merged"].items())
)
CLASS_TABLE = "\n".join(
    f"| {label} | {SUMMARY['classes'][label]['category']} | {SUMMARY['classes'][label]['name']} | "
    f"{SUMMARY['per_class'][label]['unique']} |"
    for label in CLASSES
)
CONFLICT_LINES = "\n".join(
    f"* `{pair}` — {jumlah} foto" for pair, jumlah in SUMMARY["conflict_pairs"].items()
)
COMMAND_STEPS = "\n".join(EXPORT_STEPS)

report = f"""# YOLOv8 classifier run — merged 18-class set (CC0 + Clothes_Dataset)

| | |
| --- | --- |
| date (UTC) | {DATE_STAMP} |
| notebook | `ml_workspace/yolo/kaggle_merged_dataset.ipynb` |
| repo ref merged in | `{REPO_REF}` |
| dataset | `merge_datasets.py` over `alexeygrigorev/clothing-dataset-small` + `ryanbadai/clothes-dataset` |
| licence of the data | CC0 for the bootstrap set; Kaggle tag MIT over scraped Carousell photographs (docs/adr/0007) |
| GPU | {GPU_NAME} |
| ultralytics / torch / python | {ultralytics.__version__} / {torch.__version__} / {sys.version.split()[0]} |
| model | `{MODEL}` (ImageNet-pretrained), fine-tuned head |
| data | {len(CLASSES)} classes, {TRAIN_IMAGES} train / {VAL_IMAGES} val / {TEST_IMAGES} test unique photographs |
| training | imgsz {IMGSZ}, {EPOCHS} epochs, batch {BATCH}, seed {SEED_TRAIN}, patience {PATIENCE}, device {DEVICE} |
| best weights | `{BEST_WEIGHTS}` |

## Measured

| split | top-1 | top-5 | accuracy | macro-F1 | images |
| --- | --- | --- | --- | --- | --- |
| val (epoch selection) | {MEASURED['val']['top1']:.4f} | {MEASURED['val']['top5']:.4f} | {MEASURED['val']['accuracy']:.4f} | {MEASURED['val']['macro_f1']:.4f} | {MEASURED['val']['images']} |
| **test (reported)** | **{TEST_TOP1:.4f}** | {MEASURED['test']['top5']:.4f} | {MEASURED['test']['accuracy']:.4f} | **{TEST_MACRO_F1:.4f}** | {MEASURED['test']['images']} |

Conversion and fidelity:

| metric | value |
| --- | --- |
| PyTorch top-1 on the {IMAGE_COUNT}-image fidelity subset (test) | {TORCH_ACCURACY:.4f} |
| `.tflite` top-1 on the same subset | {TFLITE_ACCURACY:.4f} |
| argmax agreement, `.tflite` vs PyTorch | {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f} |
| largest `abs(sum(scores) - 1)` | {MAX_SUM_ERROR:.3e} |
| largest `abs(p_tflite - p_pytorch)` | {MAX_PROB_DIFF:.4f} |
| conversion gate (`.tflite` >= 0.50 and >= PyTorch - 0.10) | {GATE_LABEL} |
| confusion-matrix cells where ultralytics and sklearn disagree (test) | {MEASURED['test']['disagreement']} |

Not comparable, and the report says so on purpose: the shipped bootstrap model is 10 classes at
91.9 % top-1 on the CC0 test split, and the public 15-class baseline is a fine-tuned ViT-base at
78.47 % accuracy / 0.7803 macro-F1 on a 40 % split. Class counts and split sizes differ.

## The merge this model was trained on

Scanned {SUMMARY['totals']['scanned']} files; {SUMMARY['totals']['kept']} unique photographs survived
({SUMMARY['totals']['groups_with_several_members']} duplicate groups). Dropped by reason:

""" + "\n".join(
    f"* `{reason}` — {jumlah}"
    for reason, jumlah in sorted(SUMMARY["totals"]["by_reason"].items(), key=lambda item: -item[1])
) + f"""

Label conflicts resolved by `--prefer-source clothes` (the surviving class is on the right):

{CONFLICT_LINES}

## Class list (this IS the model's output order)

| label | Category | Nama | unique photos |
| --- | --- | --- | --- |
{CLASS_TABLE}

CC0 folder -> merged class:

| CC0 folder | merged class |
| --- | --- |
{MAPPING_TABLE}

## Per class, split `test`

| class | precision | recall | f1-score | support |
| --- | --- | --- | --- | --- |
{per_class_table('test')}

## Per class, split `val` (for reference, used for epoch selection)

| class | precision | recall | f1-score | support |
| --- | --- | --- | --- | --- |
{per_class_table('val')}

## Confusion matrix, split `test` (rows true, columns predicted)

{matrix_table('test')}

## Known data problems (read before trusting the numbers)

* **`outwear` is a wide umbrella in CC0.** It becomes `Jaket` while Clothes_Dataset separates
  Jaket, Jaket_Denim, Jaket_Olahraga, Blazer, Mantel and Sweter, so CC0 outerwear competes with
  five finer classes. This is a modelling compromise accepted in `docs/adr/0008`.
* **403 CC0 `longsleeve` photographs carry the `Sweter` label** in Clothes_Dataset; the
  Clothes_Dataset copy won, so `Kaos_Lengan_Panjang` holds only its CC0-only photographs. If that
  class misses, the fix is `--prefer-source cc0`, not a different model.
* **Identical photos filed under two classes inside Clothes_Dataset were dropped entirely**
  (Jaket vs Kemeja vs Polo, for example), because picking one label would invent an answer the data
  does not give. `conflicts.csv` in the merge output lists each decision.

## Exported model

* `{TFLITE_PATH}`
* size {TFLITE_BYTES:,} bytes, sha256 `{TFLITE_SHA256}`
* read by `{contract.runtime}`: input `{contract.in_shape}` {contract.in_dtype.__name__}, output `{contract.out_shape}` {contract.out_dtype.__name__}, layout {contract.layout}
* class order (`classes.json` `labels`, = the model's output indices): {CLASS_LIST}

## Commands

```bash
pip install -q ultralytics
git clone --depth 1 {REPO_URL} gayaku && git -C gayaku checkout {REPO_REF}
git clone --depth 1 {CC0_URL} clothing-dataset-small
python gayaku/ml_workspace/scripts/merge_datasets.py --cc0-dir clothing-dataset-small \\
  --clothes-dir /kaggle/input/clothes-dataset/Clothes_Dataset --out merged --force
YOLO('{MODEL}').train(data=merged, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, seed={SEED_TRAIN})
YOLO('{BEST_WEIGHTS}').val(data=merged, split='test', imgsz={IMGSZ})
{COMMAND_STEPS}
```

## Licence

Trained with Ultralytics YOLOv8 on a merge of `alexeygrigorev/clothing-dataset-small` (CC0-1.0) and
`ryanbadai/clothes-dataset`. Ultralytics' licence page states that models trained with their code
are AGPL-3.0 by default, and these weights derive from the `{MODEL}` checkpoint: shipping a build
that contains this `.tflite` triggers AGPL-3.0's source obligation, and this repository has no
LICENSE file (`docs/adr/0006`). The Carousell photographs are not ours to redistribute
(`docs/adr/0007`).
"""

(ARTIFACTS_DIR / "report.md").write_text(report, encoding="utf-8")
print(f"ditulis       {ARTIFACTS_DIR / 'report.md'}")

## 8. Bungkus dan unduh

Satu zip di **`/kaggle/working`** — folder yang tersimpan sebagai output notebook (maks 20 GB):

* `garment_classifier.tflite` — modelnya, dengan sha256 dan ukuran dicetak di bawah;
* `classes.json` — urutan kelas yang dipakai model itu;
* `report.md` — angka terukur, perintah, catatan noise label dan lisensi;
* `classification_report_{val,test}.{txt,csv}`, `per_class_accuracy_{val,test}.csv`,
  `confusion_matrix_{val,test}.csv`;
* `confusion_matrix_val.png` / `_test.png` dan versi `_normalized`;
* `merged_dataset.json` dan `classes_merged.json` — bukti isi dataset yang dilatih.

Setelah zip ada di komputermu, langkah integrasi ada di `ml_workspace/yolo/README.md`: salin
`.tflite` ke `flutter_app/assets/models/garment_classifier.tflite`, lalu jalankan
`gen_dart_labels.py --classes ... --model ...` pada commit yang sama.

In [ ]:
# Langkah 8/8 — bungkus artefak dan cetak angka yang dilaporkan.
import pathlib
import zipfile

ZIP_PATH = pathlib.Path("/kaggle/working/gayaku_yolo_merged.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS_DIR.iterdir()):
        if path.is_file():
            archive.write(path, path.name)

print(f"{TFLITE_PATH.name:<40}{TFLITE_BYTES:>12,} bytes")
print(f"{'sha256':<40}{TFLITE_SHA256}")
print(f"{ZIP_PATH.name:<40}{ZIP_PATH.stat().st_size:>12,} bytes")
print()
print("Angka yang dilaporkan (split test):")
print(f"  top-1                              : {TEST_TOP1:.4f}")
print(f"  macro-F1                           : {TEST_MACRO_F1:.4f}")
print(f"  kesesuaian .tflite vs PyTorch      : {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f}")
print()
print("Selanjutnya: ml_workspace/yolo/README.md bagian integrasi ke aplikasi.")

## Catatan penutup: lisensi, dan angka yang boleh dikutip

* **Bobotnya AGPL-3.0.** Model yang dilatih dengan kode Ultralytics berlisensi AGPL-3.0, dan
  `.tflite` ini adalah karya turunan dari checkpoint `yolov8n-cls.pt`. Mengedarkan APK yang
  memuatnya berarti menawarkan karya turunan itu di bawah AGPL-3.0, dan repo ini belum punya
  berkas `LICENSE` (`docs/adr/0006-yolov8-classifier-agpl.md`). Untuk tugas kuliah, pemakaiannya
  bebas.
* **Datanya bukan milik kita.** Tag MIT di Kaggle menutupi labelnya; deskripsi dataset menyebut
  fotonya di-*scrape* dari Carousell. Bobot dan build yang memuatnya adalah artefak tugas
  (`docs/adr/0007-carousell-clothes-dataset.md`).
* **Jangan menyebut akurasi tanpa kelas dan split.** Model bootstrap 10 kelas yang sekarang
  dikirim mengukur 91,9 % top-1 pada split test CC0; run ini 18 kelas dengan split test sendiri.
  Angka keduanya tidak bisa dibandingkan langsung, dan baseline publik 15 kelas (ViT-base,
  78,47 %) memakai split 40 % yang lebih kecil support-nya per kelas.
* **Kalau `Kaos_Lengan_Panjang` menabrak `Sweter`** di confusion matrix, jawabannya bukan
  mengubah kode: jalankan `merge_datasets.py --prefer-source cc0`, lalu ulangi notebook ini.
  Itulah alasan flag-nya ada.